# **HEPTAPOD Setup & Configuration**

**Written by**: 
- Tony Menzo (U. of Alabama / Fermilab)

This notebook provides a guide to configuring HEPTAPOD for high-energy physics workflows. It covers:

0. **Install dependencies** - Setting up your Python environment
1. **config.py** - Configuring external tool paths (FeynRules, MadGraph, Pythia)
2. **Environment variables** - Setting up LLM API keys
3. **External tools** - Installing physics software dependencies
4. **Test runner** - Verifying your installation

---

**HEPTAPOD Repository**: [https://github.com/tonymenzo/heptapod](https://github.com/tonymenzo/heptapod)

## 0. Install Dependencies

We recommend creating a fresh Python environment before installing HEPTAPOD dependencies:

```bash
# Using conda
conda create -n heptapod python=3.13
conda activate heptapod

# Or using venv
python -m venv heptapod-env
source heptapod-env/bin/activate  # Linux/macOS
```

Run the cell below to install all required packages:

In [ ]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

# Install requirements
# !pip install -r {REPO_ROOT}/requirements.txt -q

## 1. `config.py` - External Tool Paths

The `config.py` file in the repository root defines paths to external physics tools. You must edit this file to match your system.

### Example config.py

```python
# config.py - HEPTAPOD Configuration

# Ollama LLM Configuration
ollama_host = None  # None = localhost:11434 (default)
# ollama_host = "http://192.168.1.100:11434"  # For remote Ollama server
ollama_model = "llama3:70b"  # Change to your preferred model

# External Dependencies
feynrules_path = "/path/to/FeynRules_v2.3.49"
wolframscript_path = "/Applications/Mathematica.app/Contents/MacOS/wolframscript"  # macOS
# wolframscript_path = "/usr/local/bin/wolframscript"  # Linux
mg5_path = "/path/to/MG5_aMC_v3.6.6"
```

In [ ]:
# View your current config.py
config_path = REPO_ROOT / "config.py"
print(f"config.py location: {config_path}")
print("=" * 70)
print(config_path.read_text())

In [ ]:
# Load and display current configuration
from config import (
    feynrules_path,
    wolframscript_path,
    mg5_path,
    ollama_host,
    ollama_model
)

print("Current configuration values:")
print("=" * 70)
print(f"feynrules_path:     {feynrules_path}")
print(f"wolframscript_path: {wolframscript_path}")
print(f"mg5_path:           {mg5_path}")
print(f"ollama_host:        {ollama_host or 'localhost:11434 (default)'}")
print(f"ollama_model:       {ollama_model}")

# Check which tools are available
print("\nTool availability:")
print("-" * 70)
tools = [
    ("FeynRules", feynrules_path),
    ("WolframScript", wolframscript_path),
    ("MadGraph5", mg5_path),
]
for name, path in tools:
    exists = Path(path).exists() if path else False
    status = "[+] Found" if exists else "[-] Not found"
    print(f"  {name:15} {status:15} {path}")

### Update config.py from this notebook

Edit the paths below and run the cell to update your `config.py` file:

In [ ]:
from examples.shared.heptapod_setup import update_config

# Edit these paths to match your system
NEW_CONFIG = {
    "feynrules_path": "/path/to/FeynRules_v2.3.49",
    "wolframscript_path": "/Applications/Mathematica.app/Contents/MacOS/wolframscript",  # macOS
    # "wolframscript_path": "/usr/local/bin/wolframscript",  # Linux
    "mg5_path": "/path/to/MG5_aMC_v3.6.6",
    "ollama_host": None,  # None = localhost:11434 (default)
    "ollama_model": "llama3:70b",
}

# Preview changes (dry run)
update_config(NEW_CONFIG, dry_run=True)

In [ ]:
# Uncomment and run to apply changes:
# update_config(NEW_CONFIG, dry_run=False)

## 2. Environment Variables (.env)

API keys for LLM providers are stored in a `.env` file in the repository root. This file is **not** committed to git.

### Supported Providers

| Variable | Provider | Model Examples |
|----------|----------|----------------|
| `OPENAI_API_KEY` | OpenAI | GPT-4o, GPT-4o-mini |
| `ANTHROPIC_API_KEY` | Anthropic | Claude Sonnet, Claude Opus |
| `GOOGLE_API_KEY` | Google | Gemini 2.0, Gemini 1.5 Pro |
| `GROQ_API_KEY` | Groq | Llama 3.3 70B (fast) |
| `MISTRAL_API_KEY` | Mistral | Mistral Large |

You only need keys for providers you plan to use. For local models, use **Ollama** instead.

In [ ]:
# Check your .env configuration
import os
from dotenv import load_dotenv

env_path = REPO_ROOT / ".env"

if env_path.exists():
    load_dotenv(env_path)
    print(f".env found at: {env_path}")
    print("=" * 70)
    
    api_keys = {
        "OPENAI_API_KEY": "OpenAI (GPT)",
        "ANTHROPIC_API_KEY": "Anthropic (Claude)",
        "GOOGLE_API_KEY": "Google (Gemini)",
        "GROQ_API_KEY": "Groq"
    }
    
    print("\nAPI Key Status:")
    for key, name in api_keys.items():
        value = os.getenv(key)
        if value:
            # Show first/last few chars for verification
            masked = f"{value[:8]}...{value[-4:]}" if len(value) > 16 else "***"
            print(f"  [+] {name:25} {masked}")
        else:
            print(f"  [-] {name:25} Not configured")
else:
    print(f".env NOT found at: {env_path}")
    print("\nCreate one with:")
    print('  echo "OPENAI_API_KEY=sk-..." > .env')

### Update .env from this notebook

Edit the API keys below and run the cell to update your `.env` file:

In [ ]:
from examples.shared.heptapod_setup import update_env

# Edit your API keys here (only include the ones you have)
NEW_ENV = {
    "OPENAI_API_KEY": "",      # e.g., "sk-proj-..."
    "ANTHROPIC_API_KEY": "",   # e.g., "sk-ant-..."
    "GOOGLE_API_KEY": "",      # e.g., "AIza..."
    "GROQ_API_KEY": ""        # e.g., "gsk_..."
}

# Preview changes (dry run)
update_env(NEW_ENV, dry_run=True)

In [ ]:
# Uncomment and run to apply changes:
# update_env(NEW_ENV, dry_run=False)

## 3. External Physics Tools

HEPTAPOD integrates with several external physics tools. Not all are required - install only what you need.

### 3.1 FeynRules (BSM Model Creation)

FeynRules generates UFO models from Lagrangians for use with MadGraph.

**Requirements:**
- Mathematica (or free Wolfram Engine)
- FeynRules package

**Installation:**
1. Download from [feynrules.irmp.ucl.ac.be](https://feynrules.irmp.ucl.ac.be/)
2. Extract to a known location
3. Update `feynrules_path` in `config.py`
4. Ensure `wolframscript` is accessible (or path set in `config.py`)

### 3.2 MadGraph5_aMC (Parton-Level Events)

MadGraph generates parton-level events for hard scattering processes.

**Installation:**
1. Download from [launchpad.net/mg5amcnlo](https://launchpad.net/mg5amcnlo)
2. Extract to a known location
3. Update `mg5_path` in config.py

### 3.3 Pythia8 (General-Purpose Event Generator)

Pythia is a general-purpose Monte Carlo event generator for high-energy collisions. It handles parton showering, hadronization, underlying event, and can also generate complete events standalone.

**Installation (via pip):**
```bash
pip install pythia8mc
```

### 3.4 Sherpa (Alternative Generator)

Sherpa is an alternative general-purpose event generator.

**Installation (via pip):**
```bash
pip install sherpa-mc
```

### 3.5 Ollama (Local LLMs)

Ollama runs LLMs locally without API keys.

**Installation:**
1. Download from [ollama.ai](https://ollama.ai)
2. Install and run `ollama serve`
3. Pull a model: `ollama pull llama3:70b`

In [ ]:
# Check Python package availability
print("Python Package Status:")
print("=" * 70)

packages = [
    ("orchestral", "Orchestral AI framework"),
    ("pythia8mc", "Pythia8 (event generation)"),
    ("Sherpa", "Sherpa (event generation)"),
    ("numpy", "NumPy (numerical computing)"),
    ("matplotlib", "Matplotlib (plotting)"),
    ("openai", "OpenAI SDK"),
    ("anthropic", "Anthropic SDK"),
]

for package, description in packages:
    try:
        __import__(package)
        print(f"  [+] {description:40} Installed")
    except ImportError:
        print(f"  [-] {description:40} Not installed")

## 4. Using the Shared Setup Module

For HEPTAPOD tool notebooks, use the shared setup module for a one-liner configuration:

In [ ]:
# The recommended way to set up HEPTAPOD in tool notebooks
from examples.shared.heptapod_setup import setup_heptapod

# This displays all configuration and returns a dict
config = setup_heptapod()

# You can access config values programmatically:
# config['feynrules_path'], config['mg5_path'], etc.

In [ ]:
# For silent setup (no output), use verbose=False
config_silent = setup_heptapod(verbose=False)
print(f"Loaded config with {len(config_silent)} fields")

## 5. Test Runner

HEPTAPOD includes a comprehensive test runner (`test_runner.py`) that verifies your installation.

### Usage

```bash
# Run all tests
python test_runner.py

# Run with verbose output
python test_runner.py --verbose

# Skip slow integration tests (MG5, Pythia, Sherpa generation)
python test_runner.py --skip-slow

# Run only prerequisites check
python test_runner.py --only prereqs

# Run only specific component tests
python test_runner.py --only conversions
python test_runner.py --only kinematics
python test_runner.py --only feynrules
python test_runner.py --only mg5
python test_runner.py --only pythia
python test_runner.py --only sherpa

# Keep generated test files (useful for debugging)
python test_runner.py --keep-files
```

### Test Components

| Component | Description | Requirements |
|-----------|-------------|-------------|
| `prereqs` | Prerequisites check | None |
| `llm` | Ollama integration | Ollama running |
| `conversions` | LHE/JSONL/NumPy conversion | None |
| `kinematics` | Invariant mass, pT, cuts | None |
| `reconstruction` | Resonance reconstruction | None |
| `delta_r_filter` | Overlap removal | None |
| `feynrules` | UFO generation | FeynRules + Mathematica |
| `mg5` | Event generation | MadGraph5 |
| `pythia` | Showering/hadronization | pythia8mc |
| `sherpa` | Alternative generation | sherpa-mc |

In [ ]:
# Run prerequisites check from notebook
import subprocess

print("Running: python test_runner.py --only prereqs")
print("=" * 70)

result = subprocess.run(
    [sys.executable, str(REPO_ROOT / "test_runner.py"), "--only", "prereqs"],
    cwd=REPO_ROOT,
    capture_output=True,
    text=True
)

print(result.stdout)
if result.stderr:
    print("STDERR:", result.stderr)

In [ ]:
# Run quick tests (skip slow physics tools)
print("Running: python test_runner.py --skip-slow")
print("=" * 70)
print("This runs conversions, kinematics, reconstruction, and delta_r_filter tests.")
print("(Skips feynrules, mg5, pythia, sherpa which require external tools)")
print()

# Uncomment to run:
# result = subprocess.run(
#     [sys.executable, str(REPO_ROOT / "test_runner.py"), "--skip-slow"],
#     cwd=REPO_ROOT
# )

## Summary

To set up HEPTAPOD:

1. **Create a fresh environment** - Use conda or venv to isolate dependencies
2. **Install dependencies** - Run the pip install cell at the top of this notebook
3. **Edit `config.py`** - Set paths to FeynRules, MadGraph, WolframScript
4. **Create `.env`** - Add API keys for your LLM providers
5. **Verify installation** - `python test_runner.py --only prereqs`
6. **Run tests** - `python test_runner.py --skip-slow` (quick) or `python test_runner.py` (full)

**Next steps:**
- See `orchestral_setup_basics.ipynb` to learn the Orchestral agent framework
- See `examples/feynrules/` for FeynRules workflows
- See `examples/madgraph/` for MadGraph event generation
- See `examples/analysis/` for event analysis tools

## Troubleshooting

### Common Issues

**"config.py not found"**
- Ensure you're running from the correct directory
- Check that config.py exists in the repository root

**"FeynRules not found"**
- Verify `feynrules_path` in config.py points to the correct directory
- The path should contain `FeynRules.m`

**"WolframScript not found"**
- macOS: `/Applications/Mathematica.app/Contents/MacOS/wolframscript`
- Linux: `/usr/local/bin/wolframscript` (or use `which wolframscript`)
- You can also use the free Wolfram Engine

**"No LLM available"**
- Add API keys to `.env`, OR
- Install Ollama and run `ollama serve`

**"pythia8mc / sherpa-mc import error"**
- Ensure you're using the heptapod environment with Python 3.13
- Try reinstalling: `pip install --force-reinstall pythia8mc sherpa-mc`